In [ ]:
import pandas as pd
import numpy as np

#Load messy dataset
df = pd.read_csv('marketing_campaign_data_messy.csv')

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
df.head(3)

# Header cleaning

In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')
df.head()

# Type conversion & currency cleaning

In [ ]:
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask ,['campaign_id','spend']].head(3))

In [ ]:
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]','', regex = True)
df['spend'] = pd.to_numeric(df['spend'])

print(df.loc[dirty_spend_mask ,['campaign_id','spend']].head(3))

# Categorical Typos

In [ ]:
print(df['channel'].unique())
cleanup_map = {
    'Facebok':'Facebook',
    'Insta_gram':'Instagram',
    'Tik_Tok':'TikTok',
    'E-mail' : 'Email',
    'Gogle':'Google Ads' ,
    'N/A': np.nan
}

df['channel'] = df['channel'].replace(cleanup_map)
print(df['channel'].unique())


# Handling mixed booleans

In [ ]:
print(df['active'].unique())
bool_map = {
'Yes': True,
'True': True,
'Y': True,
'1': True,
'No': False,
'False': False,
'0': False,
'N': False
}

df['active']= df['active'].map(bool_map).fillna(False).astype(bool)
print(df['active'].unique())

# Date parsing

In [ ]:
print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'],format='mixed',dayfirst= True, errors = 'coerce')
df['end_date'] = pd.to_datetime(df['end_date'],format='mixed',dayfirst= True, errors = 'coerce')

print(df['start_date'].dtype)

# Logical Integrity (clicks vs impressions)

In [ ]:
df = df.loc[:,~df.columns.duplicated()]
impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask ,['campaign_id','impressions','clicks']].head(3))


# Logical integrity (Time travel)

In [ ]:
time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id','start_date','end_date']].head(3))

# It will be assumed that the duration of the campaign is 30 days, hence the following approach to this solution for the errors that were found
df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask,'start_date'] + pd.Timedelta(days=30)
print(df.loc[time_travel_mask, ['campaign_id','start_date','end_date']].head(3))


# Handling outliers (winsorizing)

In [ ]:
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1 
upper_limit = round(Q3 + (3 * IQR),2)

outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask, ['campaign_id','spend']].head(3))

df.loc[outlier_mask,'spend'] = upper_limit
print(df.loc[outlier_mask, ['campaign_id','spend']].head(3))

# String parsing (feature extraction)

In [ ]:
print(df['campaign_name'].head(3))
df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')
print(df[['campaign_name','season']].head(3))
